In [9]:
# Install libraries
!pip install -q gradio transformers datasets scikit-learn joblib nltk tabulate

import gradio, torch, transformers
print("gradio       :", gradio.__version__)
print("transformers :", transformers.__version__)
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no (using CPU)")

gradio       : 6.20.0
transformers : 5.13.1
GPU          : no (using CPU)


In [10]:
# Load artifacts from Google Drive
import os, json, warnings
import numpy as np
import torch
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ART = "/content/drive/MyDrive/tweeteval_demo"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 128

assert os.path.isdir(ART), f"{ART} not found. Run the export cell in the training notebook first."

LABELS = ["negative", "neutral", "positive"]
META = {}
if os.path.exists(f"{ART}/meta.json"):
    META = json.load(open(f"{ART}/meta.json"))
    LABELS = META.get("label_names", LABELS)

id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}

from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODELS = {}
LOAD_LOG = []

# 1. TF-IDF + LinearSVC
try:
    import joblib, nltk
    try:
        nltk.data.find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords', quiet=True)
    MODELS["TF-IDF + LinearSVC"] = {
        "kind": "svm",
        "vec": joblib.load(f"{ART}/tfidf_vectorizer.joblib"),
        "clf": joblib.load(f"{ART}/svm_linearsvc.joblib"),
    }
    LOAD_LOG.append("OK   TF-IDF + LinearSVC")
except Exception as e:
    LOAD_LOG.append(f"MISS TF-IDF + LinearSVC  ({e})")

# 2. BERT
try:
    MODELS["BERT (full FT)"] = {
        "kind": "hf",
        "tok": AutoTokenizer.from_pretrained(f"{ART}/bert"),
        "model": AutoModelForSequenceClassification.from_pretrained(f"{ART}/bert").to(DEVICE).eval(),
    }
    LOAD_LOG.append("OK   BERT (full FT)")
except Exception as e:
    LOAD_LOG.append(f"MISS BERT  ({e})")

# 3. RoBERTa full fine-tune
try:
    try:
        tok = AutoTokenizer.from_pretrained(f"{ART}/roberta_full")
    except Exception:
        tok = AutoTokenizer.from_pretrained("roberta-base")
    MODELS["RoBERTa (full FT)"] = {
        "kind": "hf",
        "tok": tok,
        "model": AutoModelForSequenceClassification.from_pretrained(f"{ART}/roberta_full").to(DEVICE).eval(),
    }
    LOAD_LOG.append("OK   RoBERTa (full FT)")
except Exception as e:
    LOAD_LOG.append(f"MISS RoBERTa full FT  ({e})")

print("=" * 58)
for l in LOAD_LOG:
    print(" ", l)
print("=" * 58)
print(f"Loaded {len(MODELS)}/4 models. Device = {DEVICE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  OK   TF-IDF + LinearSVC
  OK   BERT (full FT)
  OK   RoBERTa (full FT)
Loaded 3/4 models. Device = cpu


In [11]:
# Load LoRA r=32 by merging the adapter manually (no peft needed)
import re, json as _json
from safetensors.torch import load_file as _load_sft

def _norm(k):
    k = k.replace("base_model.model.", "")
    k = k.replace("modules_to_save.default.", "").replace("modules_to_save.", "")
    k = k.replace("original_module.", "")
    k = k.replace(".default.", ".")
    return k

CLS_KEYS = ("classifier.dense.weight", "classifier.dense.bias",
            "classifier.out_proj.weight", "classifier.out_proj.bias")

def load_lora_merged(adapter_dir, base_name="roberta-base"):
    cfg = _json.load(open(f"{adapter_dir}/adapter_config.json"))
    r, alpha = cfg["r"], cfg["lora_alpha"]
    scaling = alpha / (r ** 0.5) if cfg.get("use_rslora") else alpha / r

    p = f"{adapter_dir}/adapter_model.safetensors"
    sd = _load_sft(p) if os.path.exists(p) else torch.load(
        f"{adapter_dir}/adapter_model.bin", map_location="cpu")

    model = AutoModelForSequenceClassification.from_pretrained(
        base_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

    pairs, cls_sd = {}, {}
    for k, v in sd.items():
        n = _norm(k)
        m = re.match(r"^(.*)\.lora_([AB])\.weight$", n)
        if m:
            pairs.setdefault(m.group(1), {})[m.group(2)] = v
        elif n.endswith(CLS_KEYS):
            cls_sd[n[n.index("classifier.") + len("classifier."):]] = v

    params = dict(model.named_parameters())
    merged = 0
    for mod, ab in pairs.items():
        if "A" not in ab or "B" not in ab:
            continue
        wkey = f"{mod}.weight"
        if wkey not in params:
            print("   skipped (not found in base model):", wkey); continue
        A, B = ab["A"].float(), ab["B"].float()
        with torch.no_grad():
            params[wkey] += (B @ A).to(params[wkey].dtype) * scaling
        merged += 1

    res = model.classifier.load_state_dict(cls_sd, strict=False)
    print(f"   r={r}, alpha={alpha}, scaling={scaling}")
    print(f"   merged {merged} lora modules | classifier keys loaded: {len(cls_sd)}/4")
    if res.missing_keys:
        print("   classifier missing:", res.missing_keys)
    if merged == 0 or len(cls_sd) < 4:
        raise RuntimeError("Adapter mismatch, please check the lora_r32 folder")
    return model.eval()

try:
    _m = load_lora_merged(f"{ART}/lora_r32").to(DEVICE)
    MODELS["RoBERTa + LoRA (r=32)"] = {
        "kind": "hf",
        "tok": AutoTokenizer.from_pretrained("roberta-base"),
        "model": _m,
    }
    print("\nOK - RoBERTa + LoRA (r=32) loaded successfully")
except Exception as e:
    print("\nSTILL FAILING:", type(e).__name__, e)

print("Models currently available:", len(MODELS), "->", list(MODELS))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   r=32, alpha=16, scaling=0.5
   merged 24 lora modules | classifier keys loaded: 4/4

OK - RoBERTa + LoRA (r=32) loaded successfully
Models currently available: 4 -> ['TF-IDF + LinearSVC', 'BERT (full FT)', 'RoBERTa (full FT)', 'RoBERTa + LoRA (r=32)']


In [12]:
# Preprocessing + prediction
import re, time
import nltk
from nltk.corpus import stopwords

STOP = set(stopwords.words('english')) - {'not', 'no', 'nor'}

def clean_tweet(text):
    """Preprocessing for TF-IDF + SVM (same as section 3.1 of the training notebook)."""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\b(rt|qt)\b', '', text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r'\d+(st|nd|rd|th)\b', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(w for w in text.split() if w not in STOP)

def clean_tweet_bert(text):
    """Preprocessing for BERT / RoBERTa (same as section 3.2 of the training notebook)."""
    text = re.sub(r'http\S+|www\S+', 'http', text)
    text = re.sub(r'@\w+', '@user', text)
    return re.sub(r'\s+', ' ', text).strip()

def _softmax(x):
    x = np.asarray(x, dtype=np.float64)
    e = np.exp(x - x.max())
    return e / e.sum()

@torch.no_grad()
def _predict_hf(entry, text):
    enc = entry["tok"](clean_tweet_bert(text), return_tensors="pt",
                       truncation=True, max_length=MAX_LEN)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    logits = entry["model"](**enc).logits[0].float().cpu().numpy()
    return _softmax(logits)

def _predict_svm(entry, text):
    X = entry["vec"].transform([clean_tweet(text)])
    scores = entry["clf"].decision_function(X)[0]
    # LinearSVC has no predict_proba, so the margin is normalized with softmax.
    # Relative display only, NOT actual probabilities.
    return _softmax(scores)

def predict_all(text):
    """Returns {model_name: (label_scores_dict, predicted_label, time_ms)}"""
    out = {}
    for name, entry in MODELS.items():
        t0 = time.time()
        probs = _predict_svm(entry, text) if entry["kind"] == "svm" else _predict_hf(entry, text)
        ms = (time.time() - t0) * 1000
        scores = {LABELS[i]: float(probs[i]) for i in range(len(LABELS))}
        out[name] = (scores, LABELS[int(np.argmax(probs))], ms)
    return out

for n, (s, lab, ms) in predict_all("i love this so much, best day ever!").items():
    print(f"{n:<24} -> {lab:<9} ({max(s.values()):.1%})  {ms:.0f} ms")

TF-IDF + LinearSVC       -> positive  (99.4%)  31 ms
BERT (full FT)           -> positive  (99.2%)  7848 ms
RoBERTa (full FT)        -> positive  (99.6%)  1825 ms
RoBERTa + LoRA (r=32)    -> positive  (99.6%)  1354 ms


In [13]:
# Demo
import gradio as gr
import pandas as pd

EMOJI = {"negative": "🔴", "neutral": "⚪", "positive": "🟢"}

# ---------- Tab 1: Freeform tweet input ----------
def run_single(text):
    if not text or not text.strip():
        empty = {l: 0.0 for l in LABELS}
        return [empty] * 4 + ["Enter a tweet and click **Analyze**."]

    res = predict_all(text)
    order = ["TF-IDF + LinearSVC", "BERT (full FT)",
             "RoBERTa (full FT)", "RoBERTa + LoRA (r=32)"]

    labels_out, votes, lines = [], [], []
    for name in order:
        if name in res:
            scores, pred, ms = res[name]
            labels_out.append(scores)
            votes.append(pred)
            lines.append(f"| {name} | {EMOJI[pred]} **{pred}** | {scores[pred]:.1%} | {ms:.0f} ms |")
        else:
            labels_out.append({l: 0.0 for l in LABELS})
            lines.append(f"| {name} | not loaded yet | | |")

    md_out = ["**Actual preprocessing applied to the model**",
              f"- SVM: `{clean_tweet(text) or '(empty after cleaning)'}`",
              f"- BERT / RoBERTa: `{clean_tweet_bert(text)}`",
              "",
              "| Model | Prediction | Confidence | Time |",
              "|---|---|---|---|"] + lines + [""]

    if votes:
        uniq = set(votes)
        if len(uniq) == 1:
            md_out.append(f"### All {len(votes)} models agree: **{votes[0]}**")
        else:
            dist = ", ".join(f"{v}: {votes.count(v)}" for v in sorted(uniq))
            md_out.append(f"### ⚠️ Models **disagree** ({dist})")
            md_out.append("Disagreement often occurs with ambiguous or context-lacking tweets, "
                          "or those with negation, aligning with the dominant reasons "
                          "in the Error Analysis section of the report.")
    return labels_out + ["\n".join(md_out)]

EXAMPLES = [
    ["I absolutely love this new update, best thing they have shipped all year!"],
    ["the concert last night was a complete disaster, waste of money"],
    ["Apple will announce its quarterly earnings on Thursday at 5pm ET."],
    ["Great, another Monday meeting that could have been an email."],
    ["@user this is not bad at all, actually pretty impressed"],
    ["I don't hate it, but I wouldn't recommend it either"],
    ["cant wait for the exam tomorrow /s"],
    ["@user check this out http"],
]

# ---------- Tab 2: Shared errors ----------
ERR_DF = None
try:
    p = f"{ART}/common_errors_all_three_models.csv"
    if os.path.exists(p):
        ERR_DF = pd.read_csv(p)
        print(f"Shared errors loaded: {len(ERR_DF)} tweets")
except Exception as e:
    print("Could not load shared errors file:", e)

def run_shared_error():
    if ERR_DF is None:
        return ("The file `common_errors_all_three_models.csv` is not in the Drive folder. "
                "Run the export cell in the training notebook again to copy it.")
    row = ERR_DF.sample(1).iloc[0]
    res = predict_all(row["text"])
    lines = [f"> {row['text']}", "",
             f"**Ground Truth: {EMOJI.get(row['true_label'], '')} {row['true_label']}**", "",
             "| Model | Current Prediction | Reported Prediction |", "|---|---|---|"]
    keymap = {"TF-IDF + LinearSVC": "predicted_label_svm",
              "BERT (full FT)": "predicted_label_bert",
              "RoBERTa (full FT)": "predicted_label_roberta"}
    for name, (scores, pred, ms) in res.items():
        old = row.get(keymap.get(name, ""), "N/A")
        lines.append(f"| {name} | {EMOJI[pred]} {pred} ({scores[pred]:.0%}) | {old} |")
    if "primary_reason" in row:
        lines += ["", f"### Assigned Reason: **{row['primary_reason']}**"]
    return "\n".join(lines)

# ---------- Interface ----------
with gr.Blocks(title="TweetEval Sentiment Demo", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# Twitter Sentiment Classification Demo\n"
        "**Topic 8 · INS 3080 Artificial Intelligence · International School, VNU Hanoi**\n\n"
        "Direct comparison of 4 methods on the same tweet: vocabulary baseline, "
        "two full fine-tuned Transformers, and the parameter-efficient LoRA version."
    )

    with gr.Tab("1 · Analyze tweet"):
        inp = gr.Textbox(label="Enter tweet (English)", lines=3,
                         placeholder="e.g. I love this so much, best day ever!")
        btn = gr.Button("Analyze", variant="primary")
        with gr.Row():
            o1 = gr.Label(label="① TF-IDF + LinearSVC: 56.68%", num_top_classes=3)
            o2 = gr.Label(label="② BERT full FT: 67.60%", num_top_classes=3)
        with gr.Row():
            o3 = gr.Label(label="③ RoBERTa full FT: 71.06%", num_top_classes=3)
            o4 = gr.Label(label="④ RoBERTa + LoRA r=32: 71.29%", num_top_classes=3)
        detail = gr.Markdown()
        btn.click(run_single, inp, [o1, o2, o3, o4, detail])
        inp.submit(run_single, inp, [o1, o2, o3, o4, detail])
        gr.Examples(EXAMPLES, inp, label="Provided Examples")

    with gr.Tab("2 · Shared errors"):
        gr.Markdown(
            "1,605 tweets (13.07% of the test set) where **all three** architectures failed. "
            "The main causes are **ambiguous labels (42.0%)** and **lack of context (33.7%)**, "
            "not sarcasm (0.2%) as often assumed."
        )
        b2 = gr.Button("Pick a shared error", variant="primary")
        r2 = gr.Markdown()
        b2.click(run_shared_error, None, r2)

demo.launch(share=True, debug=False)

Shared errors loaded: 1605 tweets
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://490f298d1a4da09b6f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
